In [ ]:
from snowflake.ml.registry import Registry 
from snowflake.snowpark.context import get_active_session
import pandas as pd
import transformers



session = get_active_session()


reg = Registry(
    session=session,
    database_name="DOCS_DB",   # e.g. "ML_DB"
    schema_name="MAIN", # e.g. "MODEL_REGISTRY"
)

In [ ]:

ner_pipeline = transformers.pipeline(
    task="token-classification",
    model="OpenMed/OpenMed-NER-DiseaseDetect-SuperClinical-434M",
    aggregation_strategy="simple", 
    device_map='auto'
)


In [ ]:
input_data = [
"Ms. Johnson is a gastroenterology patient seen on 2025-09-01. She denies chest pain but endorses night sweats in the setting of generalized anxiety disorder. Further imaging is recommended to evaluate progression."
, "Mr. Patel is a oncology patient seen on 2025-09-12. She denies chest pain but endorses night sweats in the setting of type 2 diabetes mellitus."
]


In [ ]:

# Log the model with pip dependencies
ner_model = reg.log_model(
    ner_pipeline,
    model_name="ner_openmed",
    version_name="v1",
    sample_input_data=input_data,  # Needed for determining signature of the model
    pip_requirements=["sentence-transformers", "torch", "transformers"], # If you want to run this model in the Warehouse, you can use conda_dependencies instead,
    options={"use_gpu": True}
)

In [ ]:
# Deploy the model to SPCS
# 1 gpu_request and 4 max_instances will force to use all 4 gpus
ner_model.create_service(
    service_name="ner_openmed_svc",
    service_compute_pool="GPU_ML_M_POOL",  # Using GPU_NV_M - MEDIUM COMPUTE POOL WITH 4 GPUS
    ingress_enabled=True,
    gpu_requests="4", 
    max_instances=4,
    num_workers=4,
    max_batch_rows=64
)

In [ ]:
df = session.sql('SELECT TEXT AS "inputs", * FROM DOCS_DB.MAIN.SYNTH_DISEASE_NOTES LIMIT 10').to_pandas()


In [ ]:
model = reg.get_model("ner_openmed")   # model name from UI
mv = model.version("v1")      

In [ ]:
df_out = mv.run(
    df[['inputs']],
    function_name="__call__",   # HF pipeline entrypoint
    service_name="ner_openmed_svc",  
)

df_out